In [35]:
#imports
from openai import OpenAI
from scraper import fetch_website_contents,fetch_website_links
import json
from IPython.display import Markdown,display,update_display
from urllib.parse import urljoin, urlparse

In [36]:
# setting up the environment
ollama = OpenAI(base_url="http://localhost:11434/v1",api_key="ollama")

In [37]:
# system prompt for link
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevent to include in a brochure about a company,
such as links to an about page, or a company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links":[
        {"type":"about page", "url": "https://whatever/about"}
        {"type": "career page", "url": "https://whatever/career"}
    ]
}
"""

In [38]:
# user prompt for links
def get_links_user_prompt(url):
    user_prompt = f"""
here is the list of links on the website {url}
please decide which of these are the relevent web links for a brochure about the company,
respond with the full https URL in JSON format
Do not include Terms of Service, Privacy, email links.

Links(some might be relative links):

    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [39]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model="gemma3",
        messages=[
            {"role":"system","content":link_system_prompt},
            {"role":"user","content":get_links_user_prompt(url)}
        ],
        response_format={"type":"json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
     # Fix any relative URLs the model returned
    base = f"{urlparse(url).scheme}://{urlparse(url).netloc}"
    for link in links.get("links", []):
        if not link["url"].startswith("http"):
            link["url"] = urljoin(base, link["url"])
    
    # Filter out anything that's still not a valid URL
    links["links"] = [l for l in links["links"] if l["url"].startswith("http")]
    
    print(f"Found {len(links['links'])} relevant links")
    return links

In [44]:
def fetch_page_and_all_relevant_links(url):
    content = fetch_website_contents(url)
    relevant_link = fetch_website_links(url)
    result = f"## Landing Page:\n\n{content}\n## Relevant Links:\n"
    for link in relevant_link['links']:
        result += f"\n\n### Link:{link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [45]:
print(fetch_page_and_all_relevant_links("https://coursera.org"))

TypeError: list indices must be integers or slices, not str

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))